# SUDO KG MCP Tool Playground

Interactive notebook for testing every `skg_mcp` tool against the live SUDO KG backend.

It starts the MCP server over stdio with `SKG_BACKEND=sudo_kg`, loads the repo `.env`, maps your Fuseki/Milvus/LM Studio settings into `SKG_*` variables, then lets you call each tool independently.

In [1]:
import asyncio
import json
import os
import sys
from pathlib import Path
from typing import Any

from skg_mcp.client import SKGMCPClient
from skg_mcp.models import (
    ConceptFilter,
    ExpandContextArgs,
    ExpandNeighborsArgs,
    FilterPapersArgs,
    GetAttributionArgs,
    GetProvenanceArgs,
    LexicalSearchArgs,
    NodeKind,
    PaperFilter,
    ResolveConceptReferenceArgs,
    SearchNodeType,
    SemanticConstraintSearchArgs,
    SemanticSearchArgs,
    SparqlTemplateFilter,
    StatementFilter,
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "server.py").exists() and (PROJECT_ROOT.parent / "src" / "server.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def pretty(obj: Any) -> None:
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump(mode="json")
    print(json.dumps(obj, indent=2, ensure_ascii=False))

PROJECT_ROOT

PosixPath('/home/nipdep/Dev/skg-mcp')

## Environment

This cell loads `.env` and translates the connection details into the names consumed by `SudoKGBackend`. It intentionally does not print secrets.

In [2]:
def load_dotenv(path: Path) -> dict[str, str]:
    values = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values

def join_host_port(host: str | None, port: str | None) -> str | None:
    if not host:
        return None
    base = host.strip().rstrip("/")
    if not port or base.rsplit(":", 1)[-1].isdigit():
        return base
    return f"{base}:{port.strip()}"

def build_stdio_env() -> dict[str, str]:
    env = dict(os.environ)
    env.update(load_dotenv(PROJECT_ROOT / ".env"))
    env["SKG_BACKEND"] = "sudo_kg"
    env.setdefault("SKG_FUSEKI_URL", join_host_port(env.get("FUSEKI_HOST"), env.get("FUSEKI_PORT")) or "")
    env.setdefault("SKG_FUSEKI_DATASET", env.get("FUSEKI_DATASET_NAME", "sudo_kg"))
    env.setdefault("SKG_MILVUS_URI", join_host_port(env.get("MILVUS_HOST"), env.get("MILVUS_PORT")) or "")
    env.setdefault("SKG_MILVUS_COLLECTION", env.get("MILVUS_COLLECTION_NAME", "sudo_kg"))
    env.setdefault("SKG_MILVUS_DB_NAME", env.get("MILVUS_DB_NAME", ""))
    env.setdefault("SKG_EMBEDDER_PROVIDER", env.get("MODEL_PROVIDER", "openai"))
    env.setdefault("SKG_EMBEDDER_BASE_URL", env.get("PROVIDER_URL", env.get("OPENAI_BASE_URL", "")))
    env.setdefault("SKG_EMBEDDER_API_KEY", env.get("OPENAI_API_KEY", "lmstudio"))
    env.setdefault("SKG_EMBEDDER_MODEL", env.get("EMBEDDING_MODEL", env.get("JINA_MODEL", "jina-v4-text-matching")))
    env.setdefault("SKG_BACKEND_TIMEOUT_SECONDS", "60")
    return {key: value for key, value in env.items() if value != ""}

STDIO_ENV = build_stdio_env()
safe_env_preview = {k: v for k, v in STDIO_ENV.items() if k.startswith("SKG_") and "KEY" not in k and "TOKEN" not in k}
pretty(safe_env_preview)

{
  "SKG_BACKEND": "sudo_kg",
  "SKG_FUSEKI_URL": "http://sudo-kg.idea.rpi.edu:3030",
  "SKG_FUSEKI_DATASET": "gold_standard_kg",
  "SKG_MILVUS_URI": "http://spark-6d47.tailb1f37b.ts.net:19531",
  "SKG_MILVUS_COLLECTION": "gold_std_abstract_index",
  "SKG_MILVUS_DB_NAME": "sudo_graph_db",
  "SKG_EMBEDDER_PROVIDER": "openai",
  "SKG_EMBEDDER_BASE_URL": "https://spark-6d47:12345/v1",
  "SKG_EMBEDDER_MODEL": "jina-v4-text-matching",
  "SKG_BACKEND_TIMEOUT_SECONDS": "60"
}


## Connect And Inspect

In [5]:
async with SKGMCPClient.connect_stdio(
    env=STDIO_ENV,
    args=["run", "src/server.py"],
    cwd=str(PROJECT_ROOT),
    read_timeout_seconds=120,
) as client:
    metadata = await client.server_metadata()
    tools = await client.list_tools()

pretty(metadata)
print([tool.name for tool in tools])

{
  "name": "Scholarly Knowledge Graph MCP",
  "version": "0.3.0",
  "tool_count": 9,
  "tool_names": [
    "filter_papers",
    "lexical_search",
    "semantic_search",
    "semantic_constraint_search",
    "resolve_concept_reference",
    "expand_context",
    "expand_neighbors",
    "get_attribution",
    "get_provenance"
  ],
  "resources": [
    "skg://metadata/server",
    "skg://catalog/tools"
  ],
  "resource_templates": [
    "skg://catalog/tools/{tool_name}"
  ]
}
['filter_papers', 'lexical_search', 'semantic_search', 'semantic_constraint_search', 'resolve_concept_reference', 'expand_context', 'expand_neighbors', 'get_attribution', 'get_provenance']


## Shared Test State

Run the cells top-to-bottom once to populate `paper_id`, `concept_id`, and `statement_id`. After that, edit any values manually and rerun individual tool cells.

In [6]:
QUERY = "eventuality entailment graph"
paper_id = None
concept_id = None
concept_label = None
statement_id = None
statement_text = None

QUERY

'eventuality entailment graph'

## `filter_papers`

In [7]:
async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    filter_papers_result = await client.filter_papers(FilterPapersArgs(limit=5))

pretty(filter_papers_result)
if filter_papers_result.papers:
    paper_id = filter_papers_result.papers[0].id
paper_id

{
  "papers": [
    {
      "id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "title": "Enriching Large-Scale Eventuality Knowledge Graph with Entailment Relations",
      "year": null,
      "venue": null
    },
    {
      "id": "revisiting_evaluation_of_knowledge_base_completion_models",
      "title": "Revisiting Evaluation of Knowledge Base Completion Models",
      "year": null,
      "venue": null
    },
    {
      "id": "learning_credal_sum_product_networks",
      "title": "Learning Credal Sum Product Networks",
      "year": null,
      "venue": null
    },
    {
      "id": "syntactic_question_abstraction_and_retrieval_for_data_scarce_semantic_parsing",
      "title": "Syntactic Question Abstraction and Retrieval for Data-Scarce Semantic Parsing",
      "year": null,
      "venue": null
    },
    {
      "id": "ranking_vs_classifying_measuring_knowledge_base_completion_quality",
      "title": "Ranking vs. Classifying: Measuring Kno

'enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations'

## `lexical_search`

In [8]:
async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    lexical_result = await client.lexical_search(
        LexicalSearchArgs(
            query=QUERY,
            node_types=[SearchNodeType.CONCEPT, SearchNodeType.STATEMENT],
            limit=6,
        )
    )

pretty(lexical_result)
if lexical_result.concepts:
    concept_id = lexical_result.concepts[0].id
    concept_label = lexical_result.concepts[0].label
if lexical_result.statements:
    statement_id = lexical_result.statements[0].id
    statement_text = lexical_result.statements[0].text

concept_id, concept_label, statement_id

{
  "concepts": [
    {
      "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
      "label": "three-step eventuality entailment graph construction method",
      "aliases": null,
      "concept_type": "Method",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    },
    {
      "id": "0d08976a-6701-4aa1-a17b-e403c32c4d1b",
      "label": "eventuality entailment graph construction task",
      "aliases": null,
      "concept_type": "Artifact",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    }
  ],
  "statements": [
    {
      "id": "127229c1-9c7f-4f68-9e75-0a8730b9b925",
      "text": "large-scale eventuality entailment graph",
      "statement_type": "artifact",
      "rhetorical_role": "artifac

('4ce42fca-be73-4a58-a28b-ec232acb77b0',
 'three-step eventuality entailment graph construction method',
 '127229c1-9c7f-4f68-9e75-0a8730b9b925')

## `semantic_search`

This uses the OpenAI-compatible embedder module. If the LM Studio endpoint is unavailable, the backend falls back to graph lexical search so the tool remains testable.

In [9]:
async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    semantic_result = await client.semantic_search(
        SemanticSearchArgs(
            query=QUERY,
            node_types=[SearchNodeType.CONCEPT, SearchNodeType.STATEMENT],
            limit=6,
        )
    )

pretty(semantic_result)

{
  "concepts": [
    {
      "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
      "label": "three-step eventuality entailment graph construction method",
      "aliases": null,
      "concept_type": "Method",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    },
    {
      "id": "0d08976a-6701-4aa1-a17b-e403c32c4d1b",
      "label": "eventuality entailment graph construction task",
      "aliases": null,
      "concept_type": "Artifact",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    }
  ],
  "statements": [
    {
      "id": "02c30181-327c-4cf0-923a-e89042f5a088",
      "text": "In this paper, to address the limitations of existing approaches, we propose a three-step eventuality entailment

## `semantic_constraint_search` With SPARQL Template Filter

The fragment below is inserted into the query and can use `?node`, `?paper`, `?type`, and `?label` depending on the tool path.

In [10]:
concept_iri = f"https://purl.org/twc/sudo/kg/concept/{concept_id}" if concept_id else None

statement_filters = None
if concept_iri:
    statement_filters = StatementFilter(
        sparql_filters=[
            SparqlTemplateFilter(
                where="?node sudo:mentions ?concept . FILTER(?concept IN ({{concepts}}))",
                params={"concepts": [concept_iri]},
                graph="sudo",
            )
        ]
    )

async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    constrained_result = await client.semantic_constraint_search(
        SemanticConstraintSearchArgs(
            query=QUERY,
            node_types=[SearchNodeType.STATEMENT],
            statement_filters=statement_filters,
            limit=5,
        )
    )

pretty(constrained_result)

{
  "concepts": [],
  "statements": []
}


## `resolve_concept_reference`

In [11]:
mention = concept_label or QUERY
context_text = statement_text or QUERY

async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    resolution_result = await client.resolve_concept_reference(
        ResolveConceptReferenceArgs(
            mention=mention,
            context_text=context_text,
            paper_id=paper_id,
            limit=3,
        )
    )

pretty(resolution_result)

{
  "resolutions": [
    {
      "concept": {
        "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
        "label": "three-step eventuality entailment graph construction method",
        "aliases": null,
        "concept_type": "Method",
        "is_canonical": false,
        "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
        "score": null,
        "summary": null,
        "provenance": null
      },
      "confidence": 0.75,
      "rationale": null
    }
  ]
}


## Pick A Seed Node For Expansion/Provenance

In [12]:
node_id = concept_id or statement_id
node_kind = NodeKind.CONCEPT if concept_id else NodeKind.STATEMENT

node_id, node_kind, paper_id

('4ce42fca-be73-4a58-a28b-ec232acb77b0',
 <NodeKind.CONCEPT: 'concept'>,
 'enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations')

## `expand_context`

In [13]:
async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    context_result = await client.expand_context(
        ExpandContextArgs(
            node_id=node_id,
            node_kind=node_kind,
            paper_id=paper_id,
            max_linked_nodes=5,
            max_neighbor_nodes=5,
        )
    )

pretty(context_result)

{
  "node": {
    "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
    "kind": "concept",
    "label": "three-step eventuality entailment graph construction method",
    "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
    "score": null
  },
  "artifacts": [],
  "propositions": [
    {
      "id": "02c30181-327c-4cf0-923a-e89042f5a088",
      "kind": "statement",
      "label": "In this paper, to address the limitations of existing approaches, we propose a three-step eventuality entailment graph construction method.",
      "node_type": "Approach",
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "relation": "supports"
    }
  ]
}


## `expand_neighbors`

In [14]:
async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    neighbors_result = await client.expand_neighbors(
        ExpandNeighborsArgs(
            node_id=node_id,
            node_kind=node_kind,
            paper_id=paper_id,
            hop_count=1,
            limit=10,
        )
    )

pretty(neighbors_result)

{
  "source_node": {
    "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
    "kind": "concept",
    "label": "three-step eventuality entailment graph construction method",
    "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
    "score": null
  },
  "hop_count": 1,
  "neighbors": [
    {
      "relation_type": "supports",
      "target_id": "02c30181-327c-4cf0-923a-e89042f5a088",
      "target_kind": "statement",
      "target_label": "In this paper, to address the limitations of existing approaches, we propose a three-step eventuality entailment graph construction method.",
      "score": null,
      "hop": 1
    }
  ]
}


## `get_attibution`

In [15]:
async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    attribution_result = await client.get_attribution(
        GetAttributionArgs(
            node_ids=[node_id],
            node_kind=node_kind,
        )
    )

pretty(attribution_result)


{
  "attributions": [
    {
      "node_id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
      "paper": {
        "id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
        "title": "Enriching Large-Scale Eventuality Knowledge Graph with Entailment Relations",
        "year": null,
        "venue": null
      },
      "location": {
        "section_id": "b936da1c-3c78-49ff-a786-53a22b84584c",
        "section_title": "Introduction",
        "paragraph_id": "8db20798-a2d6-4636-bd0f-88ecf9c7ffce",
        "sentence_id": "423148de-ae52-49fe-a5b9-cb0a12e7fcbc",
        "sentence_text": null
      }
    }
  ]
}


## `get_provenance`

In [16]:
async with SKGMCPClient.connect_stdio(env=STDIO_ENV, args=["run", "src/server.py"], cwd=str(PROJECT_ROOT), read_timeout_seconds=120) as client:
    provenance_result = await client.get_provenance(
        GetProvenanceArgs(
            node_ids=[node_id],
            node_kind=node_kind,
        )
    )

pretty(provenance_result)


{
  "provenance": [
    {
      "node_id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
      "provenance": [
        {
          "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
          "sentence_id": "423148de-ae52-49fe-a5b9-cb0a12e7fcbc"
        }
      ]
    }
  ]
}


## Run All Tool Calls In One Pass

In [ ]:
from scripts.test_sudo_kg_tools import run_tests

results = await run_tests(query=QUERY, read_timeout_seconds=120)
pretty([result.__dict__ for result in results])